# Random Forest

Il Random Forest è un algoritmo di machine learning supervisionato basato su una collezionei alberi decisionali indipendenti, combinati per migliorare la accuratezza e la robustezza rispetto a singoli alberi.

In [47]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, make_scorer
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Treining

In [ ]:
def training(file_path, csv_name):

    df = pd.read_csv(file_path)
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']
    df_validi = df.dropna(subset=original_target_list).copy()

    # trasformo tutto in binario altrimenti ho Nan come risultato
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    features = features.fillna(features.mean())

    # Instanzio i metodi
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    cv = GroupKFold(n_splits=5)
    
    scores = []

    # Eseguo il ciclo di cross-validation a "mano"
    for fold, (train_index, test_index) in enumerate(cv.split(features, target, groups)):
        X_train, X_test = features.iloc[train_index], features.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]


        # TODO: Da rimuovere if
        # Controlla se ogni target nel set di test ha almeno 2 classi (0 e 1).
        is_fold_valid = all(y_test[col].nunique() >= 2 for col in y_test.columns)
        
        if not is_fold_valid:
            # Se la fold è invalida, non si può calcolare uno score F1 significativo.
            # Assegno 0 e passo alla prossima fold per evitare errori.
            print(f"ATTENZIONE: Fold {fold} del file {csv_name} è invalida e viene assegnato score 0.")
            scores.append(0.0)
            continue

        # Se la fold è valida, procedo normalmente
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        
        score = f1_score(y_test, y_pred, average='micro', zero_division=0)
        scores.append(score)

    scores = np.array(scores)

    return {
        'mean_score': scores.mean(),
        'std_score': scores.std(),
        'scores_per_fold': scores
    }


# Lettura dei file

In [49]:
results = {}
print("="*50 + "\nRandom Forest con k-fold=5\n" + "="*50)
for name, file_path in datasets.items():
    results[name] = training(file_path, name)

# Stampo i risultati 
for name, metrics in results.items():
    # Estraggo i 5 punteggi per il modello corrente
    scores_per_fold = metrics['scores_per_fold']
    
    # Formatto i punteggi in una stringa pulita
    formatted_scores = [f'{s:.3f}' for s in scores_per_fold]
    
    # Stampo la riga per il modello corrente
    print(f"\nNome CSV: {name}")
    print(f"    5 punteggi (F1-Score): {formatted_scores}")
    print(f"    Media e Dev. Std.: {metrics['mean_score']:.3f} ± {metrics['std_score']:.3f}")


Random Forest con k-fold=5

Nome CSV: t2_medsam
    5 punteggi (F1-Score): ['0.815', '0.720', '0.756', '0.833', '0.619']
    Media e Dev. Std.: 0.749 ± 0.076

Nome CSV: t2_preprocessed
    5 punteggi (F1-Score): ['0.792', '0.750', '0.792', '0.744', '0.636']
    Media e Dev. Std.: 0.743 ± 0.057

Nome CSV: t2_original
    5 punteggi (F1-Score): ['0.808', '0.792', '0.809', '0.714', '0.651']
    Media e Dev. Std.: 0.755 ± 0.062

Nome CSV: medsam_dynamic
    5 punteggi (F1-Score): ['0.792', '0.750', '0.735', '0.720', '0.585']
    Media e Dev. Std.: 0.717 ± 0.070

Nome CSV: preprocessed_dynamic
    5 punteggi (F1-Score): ['0.815', '0.708', '0.723', '0.846', '0.564']
    Media e Dev. Std.: 0.731 ± 0.099

Nome CSV: original_dynamic
    5 punteggi (F1-Score): ['0.764', '0.735', '0.682', '0.816', '0.619']
    Media e Dev. Std.: 0.723 ± 0.068
